## 부분 시퀀스(Sub-sequence) Augmentation 검증 노트북

이 노트북은 `train_gru.py`에 적용된 Data Augmentation이 정상적으로 동작하는지, 특히 **마지막 좌표가 (0,0,0)이 되는지**, 그리고 **좌측 패딩(Left-Padding)이 의도대로 적용되었는지** 확인하기 위해 작성되었습니다.

In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# train_gru 파일에서 Config와 MosquitoDataset을 불러옵니다.
from train_gru import Config, MosquitoDataset

In [2]:
# 1. 데이터 로드 설정
train_files = sorted(list(Config.train_dir.glob('TRAIN_*.csv')))
train_labels = pd.read_csv(Config.train_labels_path)

# 확인을 위해 처음 1개의 파일만 로드합니다.
test_files = train_files[:1]
print(f"원본 파일: {test_files[0].name}")

원본 파일: TRAIN_00001.csv


In [3]:
# 2. 데이터셋 생성 (subseq_aug=True 활성화)
dataset = MosquitoDataset(
    test_files, 
    train_labels, 
    is_train=True, 
    use_delta=False,      # 변위(delta)가 아닌 실제 좌표를 눈으로 직접 확인하기 위해 False로 설정
    use_rotation=True,    # 회전 정규화 및 원점 이동 활성화
    subseq_aug=True       # 부분 시퀀스 증강 활성화
)

print(f"Augmentation 전 1개 파일의 궤적 수: 1개")
print(f"Augmentation 후 생성된 궤적 수: {len(dataset)}개")

Loading data: 100%|██████████| 1/1 [00:00<00:00, 210.72it/s]

캐시 저장: data\.cache\1_TRAIN_00001_TRAIN_00001.npz


Augmentation 전 1개 파일의 궤적 수: 1개
Augmentation 후 생성된 궤적 수: 46개


In [4]:
# 3. 결과 검증 출력
np.set_printoptions(formatter={'float': '{: 0.4f}'.format})

# 생성된 46개 중 다양한 길이를 가진 3개만 확인해봅니다.
for i in [0, 10, 20]:
    seq, target = dataset[i]
    
    print(f"\n{'='*50}")
    print(f"📌 [Augmented Sequence Index: {i}]")
    print(f"{'='*50}")
    print("🔹 궤적 좌표 (Sequence Coords) - shape:", seq.shape)
    print(seq.numpy())
    
    print("\n🔹 타겟(Target) 좌표:", target.numpy())
    
    # 핵심 검증 1: 마지막 좌표가 [0, 0, 0] 인가?
    last_coord = seq[-1].numpy()
    is_origin = np.allclose(last_coord, 0, atol=1e-5)
    print(f"\n✅ [검증 1] 마지막 좌표가 [0, 0, 0] 인가요? -> {is_origin}")
    
    # 핵심 검증 2: 패딩된 부분 확인 (앞부분이 동일한 값으로 채워졌는지)
    # seq[0]과 seq[1]이 같다면 패딩이 들어간 것입니다.
    is_padded_same = np.allclose(seq[0].numpy(), seq[1].numpy(), atol=1e-5)
    print(f"✅ [검증 2] 패딩 적용 여부 (첫 두 좌표 동일 여부)? -> {is_padded_same}")
    
    if is_padded_same:
        print("    -> (해설) 좌측에 동일한 좌표가 반복되므로, 나중에 use_delta=True를 쓰면 변위가 [0,0,0]이 되어 모델이 완벽하게 인식합니다.")


📌 [Augmented Sequence Index: 0]
🔹 궤적 좌표 (Sequence Coords) - shape: torch.Size([11, 3])
[[-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [-0.0537  0.0000  0.0000]
 [ 0.0000  0.0000  0.0000]]

🔹 타겟(Target) 좌표: [ 0.1076 -0.0008 -0.0000]

✅ [검증 1] 마지막 좌표가 [0, 0, 0] 인가요? -> True
✅ [검증 2] 패딩 적용 여부 (첫 두 좌표 동일 여부)? -> True
    -> (해설) 좌측에 동일한 좌표가 반복되므로, 나중에 use_delta=True를 쓰면 변위가 [0,0,0]이 되어 모델이 완벽하게 인식합니다.

📌 [Augmented Sequence Index: 10]
🔹 궤적 좌표 (Sequence Coords) - shape: torch.Size([11, 3])
[[-0.2687 -0.0068  0.0087]
 [-0.2687 -0.0068  0.0087]
 [-0.2687 -0.0068  0.0087]
 [-0.2687 -0.0068  0.0087]
 [-0.2687 -0.0068  0.0087]
 [-0.2687 -0.0068  0.0087]
 [-0.2151 -0.0050  0.0059]
 [-0.1614 -0.0048  0.0016]
 [-0.1076 -0.0022  0.0002]
 [-0.0539 -0.0000  0.0000]
 [ 0.0000  0.0000  0.0000]]

🔹 타겟(Target) 